# Module 03: Embedded Databases: SQLite & WAL Architecture

Hands-on lab examining WAL concurrency, synchronous disk flushes, and custom Python UDF extensions.


In [ ]:
import sqlite3
import tempfile
from pathlib import Path
print('SQLite version:', sqlite3.sqlite_version)


## 1. Configuring WAL Mode & Pragmas


In [ ]:
tmpdir = tempfile.mkdtemp()
db_path = Path(tmpdir) / 'test_wal.db'
conn = sqlite3.connect(db_path)
cur = conn.cursor()
cur.execute('PRAGMA journal_mode = WAL;')
mode = cur.fetchone()[0]
cur.execute('PRAGMA synchronous = NORMAL;')
print('Journal mode:', mode)


## 2. Testing Non-Blocking Concurrency (Readers vs Writers)


In [ ]:
cur.execute('CREATE TABLE events (id INTEGER PRIMARY KEY, msg TEXT);')
cur.execute('INSERT INTO events VALUES (1, "init");')
conn.commit()

# Open secondary read-only connection
read_conn = sqlite3.connect(db_path)
read_cur = read_conn.cursor()
read_cur.execute('SELECT * FROM events;')
print('Read from reader connection:', read_cur.fetchall())


## 3. Registering Custom Scalar Python Functions


In [ ]:
import math
def custom_hypot(a, b):
    return math.sqrt(a*a + b*b)

conn.create_function('py_hypot', 2, custom_hypot)
cur.execute('SELECT py_hypot(3.0, 4.0);')
print('Hypotenuse (3, 4):', cur.fetchone()[0])


## 4. Custom Stateful Aggregator


In [ ]:
class ProductAgg:
    def __init__(self):
        self.val = 1.0
    def step(self, v):
        if v is not None:
            self.val *= v
    def finalize(self):
        return self.val

conn.create_aggregate('product', 1, ProductAgg)
cur.execute('CREATE TABLE nums (v REAL);')
cur.executemany('INSERT INTO nums VALUES (?);', [(2.0,), (3.0,), (4.0,)])
conn.commit()   # an open write txn would block the WAL checkpoint below
cur.execute('SELECT product(v) FROM nums;')
print('Product aggregate (2*3*4):', cur.fetchone()[0])


## 5. Explicit WAL Checkpointing


In [ ]:
cur.execute('PRAGMA wal_checkpoint(PASSIVE);')
print('Checkpoint status:', cur.fetchall())
conn.close()
read_conn.close()


## Summary

SQLite in WAL mode provides high read concurrency and extensible in-process SQL execution.
